# Trout Age4 Image-Only Models

This notebook focuses only on the core research goal: predicting trout age from scale image information.

Models compared with the same fish-level split:

1. Texture-only RandomForest
   - Input: handcrafted texture features extracted from scale images.
2. CNN image-only ResNet18
   - Input: raw scale images only.
3. CNN + texture fusion
   - Input: raw scale images plus handcrafted texture features from the same scale images.

No length or weight variables are used in this notebook.

## 1. Setup

Run these notebooks first:

1. `trout_new_dataset_eda.ipynb`
2. `trout_texture_feature_extraction.ipynb` with `RUN_SAMPLE=False`

This notebook expects `feature_outputs/master_with_texture_features_full.csv`.

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
FEATURE_OUTPUT_DIR = CODE_DIR / "feature_outputs"
MODEL_OUTPUT_DIR = CODE_DIR / "model_outputs"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_TEXTURE_CSV = FEATURE_OUTPUT_DIR / "master_with_texture_features_full.csv"

SEED = 100
TEST_SIZE = 0.2
CLASS_NAMES = ["0+", "1+", "2+", "3+"]

random.seed(SEED)
np.random.seed(SEED)

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("MASTER_TEXTURE_CSV:", MASTER_TEXTURE_CSV)
print("exists:", MASTER_TEXTURE_CSV.exists())

## 2. Load Image-Derived Feature Table

Only rows with readable age labels and extracted texture features are used.

In [ ]:
if not MASTER_TEXTURE_CSV.exists():
    raise FileNotFoundError(
        f"Missing {MASTER_TEXTURE_CSV}. Run trout_texture_feature_extraction.ipynb with RUN_SAMPLE=False first."
    )

master_df = pd.read_csv(MASTER_TEXTURE_CSV)

required_cols = {"path", "scale_id", "fish_key", "age4", "has_texture_features"}
missing_cols = required_cols - set(master_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

model_df = master_df[
    master_df["age4"].notna()
    & master_df["has_texture_features"].astype(bool)
].copy()
model_df["age4"] = model_df["age4"].astype(int)
model_df["path_exists"] = model_df["path"].map(lambda p: Path(str(p)).exists())
model_df = model_df[model_df["path_exists"]].copy()

print("master_df:", master_df.shape)
print("model_df:", model_df.shape)
print("unique fish:", model_df["fish_key"].nunique())
print("age4 counts:")
display(model_df["age4"].value_counts().sort_index())
display(model_df[["scale_id", "fish_key", "age4", "path"]].head())

## 3. Texture Feature Columns

Metadata, labels, and fish morphology variables are excluded. `length_mm` and `weight_g` are intentionally not used.

In [ ]:
metadata_cols = {
    "path", "file", "scale_id", "fish_key", "river", "point", "fish_id", "image_idx", "relative_path",
    "sampling_date", "age_est", "obs", "label", "label_source", "known_bad", "age4", "id_old",
    "length_mm", "weight_g", "split", "is_readable_labeled", "is_age4_labeled", "path_exists", "has_texture_features",
}

texture_cols = [
    c for c in model_df.columns
    if c not in metadata_cols and pd.api.types.is_numeric_dtype(model_df[c])
]

if not texture_cols:
    raise ValueError("No texture feature columns found.")

print("texture feature count:", len(texture_cols))
print(texture_cols[:30])
assert "length_mm" not in texture_cols
assert "weight_g" not in texture_cols

## 4. Fish-Level Split

The same split is reused for all three models. This prevents leakage from multiple scale images belonging to the same fish.

In [ ]:
if len(model_df) < 100:
    raise ValueError("Not enough age4 rows. Make sure full texture extraction was completed.")
if model_df["fish_key"].nunique() < 20:
    raise ValueError("Not enough unique fish for fish-level split.")

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(model_df, model_df["age4"], groups=model_df["fish_key"]))
train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

fish_overlap = sorted(set(train_df["fish_key"]) & set(test_df["fish_key"]))
print("train:", train_df.shape, train_df["age4"].value_counts().sort_index().to_dict())
print("test :", test_df.shape, test_df["age4"].value_counts().sort_index().to_dict())
print("Fish overlap:", len(fish_overlap))
assert len(fish_overlap) == 0

## 5. Evaluation Helpers

In [ ]:
def evaluate_predictions(name: str, y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "support": int(len(y_true)),
    }

    print("\n===", name, "===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("Balanced accuracy:", round(metrics["balanced_accuracy"], 4))
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
    return metrics


def save_confusion_matrix(name: str, y_true, y_pred) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = MODEL_OUTPUT_DIR / f"confusion_matrix_{name}.csv"
    cm_df.to_csv(cm_path)
    print("saved:", cm_path)
    display(cm_df)
    return cm_df

results = []
predictions = {}

## 6. Model 1: Texture-Only RandomForest

This is the handcrafted texture baseline. It uses only features derived from the scale image.

In [ ]:
texture_rf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=500,
        random_state=SEED,
        class_weight="balanced_subsample",
        n_jobs=-1,
    ),
)

texture_rf.fit(train_df[texture_cols], train_df["age4"])
texture_pred = texture_rf.predict(test_df[texture_cols])
texture_metrics = evaluate_predictions("texture_only_rf", test_df["age4"], texture_pred)
save_confusion_matrix("texture_only_rf", test_df["age4"], texture_pred)

results.append(texture_metrics)
predictions["texture_only_rf"] = texture_pred

In [ ]:
rf_model = texture_rf.named_steps["randomforestclassifier"]
importance_df = pd.DataFrame({
    "feature": texture_cols,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_path = MODEL_OUTPUT_DIR / "texture_only_rf_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)
print("saved:", importance_path)
display(importance_df.head(30))

## 7. CNN Setup

The CNN models require `torch` and `torchvision`. `NUM_WORKERS=0` is used because Jupyter on HPC can crash with multiprocessing DataLoaders.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    import torchvision.models as tv_models
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from tqdm.auto import tqdm
    TORCH_AVAILABLE = True
except ImportError as exc:
    TORCH_AVAILABLE = False
    print("Torch/torchvision unavailable. CNN cells will be skipped.")
    print(exc)

if TORCH_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_CUDA = device.type == "cuda"
    NUM_WORKERS = 0
    PIN_MEMORY = USE_CUDA
    BATCH_SIZE = 64
    CNN_EPOCHS = 10
    LR = 1e-4

    def set_torch_seed(seed: int = SEED) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    set_torch_seed()
    print("device:", device)
    print("BATCH_SIZE:", BATCH_SIZE, "CNN_EPOCHS:", CNN_EPOCHS)

In [ ]:
if TORCH_AVAILABLE:
    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    class ImageOnlyDataset(Dataset):
        def __init__(self, df: pd.DataFrame, transform):
            self.df = df.reset_index(drop=True)
            self.transform = transform
            self.y = self.df["age4"].astype(int).to_numpy()

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
            img = self.transform(img)
            return img, int(self.y[idx])

    class ImageTextureDataset(Dataset):
        def __init__(self, df: pd.DataFrame, texture_array: np.ndarray, transform):
            self.df = df.reset_index(drop=True)
            self.texture_array = texture_array.astype(np.float32)
            self.transform = transform
            self.y = self.df["age4"].astype(int).to_numpy()

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
            img = self.transform(img)
            texture = torch.tensor(self.texture_array[idx], dtype=torch.float32)
            return img, texture, int(self.y[idx])

    def make_resnet18_backbone():
        try:
            weights = tv_models.ResNet18_Weights.DEFAULT
            model = tv_models.resnet18(weights=weights)
        except AttributeError:
            model = tv_models.resnet18(pretrained=True)
        model.fc = nn.Identity()
        return model

    class ImageOnlyClassifier(nn.Module):
        def __init__(self, n_classes: int = 4):
            super().__init__()
            self.backbone = make_resnet18_backbone()
            self.head = nn.Linear(512, n_classes)

        def forward(self, image):
            return self.head(self.backbone(image))

    class ImageTextureFusionClassifier(nn.Module):
        def __init__(self, n_texture: int, n_classes: int = 4):
            super().__init__()
            self.backbone = make_resnet18_backbone()
            self.texture_net = nn.Sequential(
                nn.Linear(n_texture, 64),
                nn.ReLU(),
                nn.BatchNorm1d(64),
                nn.Dropout(0.2),
                nn.Linear(64, 32),
                nn.ReLU(),
            )
            self.head = nn.Sequential(
                nn.Linear(512 + 32, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, n_classes),
            )

        def forward(self, image, texture):
            image_feat = self.backbone(image)
            texture_feat = self.texture_net(texture)
            return self.head(torch.cat([image_feat, texture_feat], dim=1))

    def class_weight_tensor(y: pd.Series) -> torch.Tensor:
        counts = y.value_counts().reindex([0, 1, 2, 3], fill_value=0).astype(float)
        weights = counts.sum() / (len(counts) * counts.clip(lower=1))
        return torch.tensor(weights.to_numpy(), dtype=torch.float32, device=device)

    print("datasets and model classes ready")

## 8. CNN Training Helpers

In [ ]:
if TORCH_AVAILABLE:
    def train_image_only(epochs: int = CNN_EPOCHS):
        train_ds = ImageOnlyDataset(train_df, train_transform)
        test_ds = ImageOnlyDataset(test_df, eval_transform)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = ImageOnlyClassifier().to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age4"]))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

        for epoch in range(epochs):
            model.train()
            losses = []
            for images, y in tqdm(train_loader, desc=f"cnn image-only {epoch + 1}/{epochs}"):
                images = images.to(device)
                y = y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images), y)
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")
        return model, test_loader

    def make_texture_arrays():
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        train_texture = scaler.fit_transform(imputer.fit_transform(train_df[texture_cols].to_numpy(dtype=np.float32)))
        test_texture = scaler.transform(imputer.transform(test_df[texture_cols].to_numpy(dtype=np.float32)))
        return train_texture, test_texture, imputer, scaler

    def train_image_texture_fusion(epochs: int = CNN_EPOCHS):
        train_texture, test_texture, imputer, scaler = make_texture_arrays()
        train_ds = ImageTextureDataset(train_df, train_texture, train_transform)
        test_ds = ImageTextureDataset(test_df, test_texture, eval_transform)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = ImageTextureFusionClassifier(n_texture=len(texture_cols)).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age4"]))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

        for epoch in range(epochs):
            model.train()
            losses = []
            for images, texture, y in tqdm(train_loader, desc=f"cnn+texture {epoch + 1}/{epochs}"):
                images = images.to(device)
                texture = texture.to(device)
                y = y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images, texture), y)
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")
        return model, test_loader

    @torch.no_grad()
    def evaluate_cnn(model, loader, name: str, fusion: bool = False):
        model.eval()
        y_true = []
        y_pred = []
        for batch in tqdm(loader, desc=f"evaluate {name}"):
            if fusion:
                images, texture, y = batch
                logits = model(images.to(device), texture.to(device))
            else:
                images, y = batch
                logits = model(images.to(device))
            pred = logits.argmax(dim=1).detach().cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(y.numpy().tolist())
        metrics = evaluate_predictions(name, y_true, y_pred)
        save_confusion_matrix(name, y_true, y_pred)
        return metrics, np.asarray(y_pred)

## 9. Model 2: CNN Image-Only ResNet18

This model uses raw scale images only. ImageNet weights are used for transfer learning.

In [ ]:
if TORCH_AVAILABLE:
    image_model, image_test_loader = train_image_only(epochs=CNN_EPOCHS)
    image_metrics, image_pred = evaluate_cnn(image_model, image_test_loader, "cnn_image_only_resnet18", fusion=False)
    results.append(image_metrics)
    predictions["cnn_image_only_resnet18"] = image_pred
else:
    print("Skipping CNN image-only model because torch is unavailable.")

## 10. Model 3: CNN + Texture Fusion

This model uses only scale-derived information: raw image pixels plus handcrafted texture features from the same image.

In [ ]:
if TORCH_AVAILABLE:
    fusion_model, fusion_test_loader = train_image_texture_fusion(epochs=CNN_EPOCHS)
    fusion_metrics, fusion_pred = evaluate_cnn(fusion_model, fusion_test_loader, "cnn_texture_fusion_resnet18", fusion=True)
    results.append(fusion_metrics)
    predictions["cnn_texture_fusion_resnet18"] = fusion_pred
else:
    print("Skipping CNN+texture fusion model because torch is unavailable.")

## 11. Final Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_path = MODEL_OUTPUT_DIR / "age4_image_only_model_results.csv"
results_df.to_csv(results_path, index=False)
print("saved:", results_path)
display(results_df)

In [ ]:
summary = {
    "n_rows": int(len(model_df)),
    "n_train_rows": int(len(train_df)),
    "n_test_rows": int(len(test_df)),
    "n_unique_fish": int(model_df["fish_key"].nunique()),
    "n_train_fish": int(train_df["fish_key"].nunique()),
    "n_test_fish": int(test_df["fish_key"].nunique()),
    "fish_overlap": int(len(set(train_df["fish_key"]) & set(test_df["fish_key"]))),
    "texture_feature_count": int(len(texture_cols)),
    "uses_length_weight": False,
}
summary_path = MODEL_OUTPUT_DIR / "age4_image_only_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("saved:", summary_path)
summary